In [1]:
import librosa
import soundfile as sf
import pandas as pd
import os
from tqdm import tqdm

In [2]:
THRESHOLD = 100
ANNOTATIONS_PATH = "../annotations.csv"
AUDIO_PATH = "../soundscape_data"
OUTPUT_PATH = "../sliced_clips"

os.makedirs(OUTPUT_PATH, exist_ok=True)

ann = pd.read_csv(ANNOTATIONS_PATH)
species_counts = ann['Species eBird Code'].value_counts()
valid_species = species_counts[species_counts >= THRESHOLD].index.tolist()
ann_filtered = ann[ann['Species eBird Code'].isin(valid_species)].reset_index(drop=True)

print(f'Total annotations to slice: {len(ann_filtered)}')
print(f'Species count: {ann_filtered["Species eBird Code"].nunique()}')

Total annotations to slice: 52456
Species count: 42


In [5]:
print('????' in valid_species)

True


In [6]:
valid_species = [s for s in valid_species if s != '????']
ann_filtered = ann[ann['Species eBird Code'].isin(valid_species)].reset_index(drop=True)
print(ann_filtered['Species eBird Code'].nunique())

41


In [8]:
print(source_path)
print(os.path.exists(source_path))
print(os.path.getsize(source_path) if os.path.exists(source_path) else 'N/A')

../soundscape_data\SSW_012_20170225_170013Z.flac
True
109265688


In [9]:
referenced_files = ann_filtered['Filename'].unique()
print(f'Files referenced in annotations: {len(referenced_files)}')

available_files = [f for f in referenced_files if os.path.exists(os.path.join(AUDIO_PATH, f))]
print(f'Files actually available: {len(available_files)}')

missing = set(referenced_files) - set(available_files)
print(f'Missing files: {len(missing)}')
print(list(missing)[:5])

Files referenced in annotations: 283
Files actually available: 283
Missing files: 0
[]


In [10]:
info = sf.info(source_path)
print(f'File duration: {info.duration} seconds')
print(f'Requested: offset={start}, duration={duration}, end={start+duration}')

File duration: 3600.0 seconds
Requested: offset=608.7, duration=36.59999999999991, end=645.3


In [12]:
rows = []
failed = []

for idx, row in tqdm(ann_filtered.iterrows(), total=len(ann_filtered)):
    source_file = row['Filename']
    start = row['Start Time (s)']
    end = row['End Time (s)']
    species = row['Species eBird Code']
    duration = end - start

    source_path = os.path.join(AUDIO_PATH, source_file)
    clip_filename = f'{species}_{idx}.wav'
    output_path = os.path.join(OUTPUT_PATH, clip_filename)

    try:
        y, sr = librosa.load(source_path, offset=start, duration=duration, sr=None)
        sf.write(output_path, y, sr)
        rows.append({
            'clip_filename': clip_filename,
            'species': species,
            'source_file': source_file,
            'path': output_path
        })
    except Exception as e:
        failed.append({'idx': idx, 'source_file': source_file, 'error': str(e)})
        continue

clips_df = pd.DataFrame(rows)
failed_df = pd.DataFrame(failed)

clips_df.to_csv('clips_metadata.csv', index=False)
failed_df.to_csv('failed_clips.csv', index=False)

print(f'Successful: {len(clips_df)}')
print(f'Failed: {len(failed_df)}')

  3%|▎         | 1439/49713 [00:09<04:45, 169.01it/s]C:\Users\Arup Sarkar\AppData\Local\Temp\ipykernel_18084\298502429.py:16: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(source_path, offset=start, duration=duration, sr=None)
C:\Users\Arup Sarkar\.conda\envs\birdclef\lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
100%|██████████| 49713/49713 [04:54<00:00, 168.81it/s]

Successful: 49682
Failed: 31


In [13]:
print(len(ann_filtered))
print(len(clips_df) + len(failed_df))

49713
49713


In [15]:
clips_df = pd.read_csv('../clips_metadata.csv')
print(clips_df.shape)
print(clips_df['species'].nunique())
print(clips_df['species'].value_counts())

(49682, 4)
41
species
rewbla     4972
bkcchi     4235
woothr     2981
grycat     2941
ovenbi1    2760
norcar     2582
amecro     2442
blujay     2267
tuftit     2126
comyel     2049
cangoo     1925
eawpew     1745
sonspa     1705
amerob     1493
amegfi     1385
reevir1    1295
comgra     1218
easpho      912
grcfly      809
rebwoo      778
balori      687
veery       677
swaspa      579
dowwoo      576
eastow      557
norfli      540
whbnut      490
scatan      490
yelwar      332
yebsap      298
cedwax      291
chswar      284
easkin      184
brncre      179
norwat      175
haiwoo      138
amewoo      134
pilwoo      127
wooduc      113
buwwar      106
moudov      105
Name: count, dtype: int64
